# RetinaGuard: 02 - Model Training & Systematic Component Ablations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Harsha-Vardan/RetinaGuard/blob/main/notebooks/02_Model_Training_and_Ablations.ipynb)
[![GitHub Repository](https://img.shields.io/badge/GitHub-RetinaGuard-blue.svg)](https://github.com/Harsha-Vardan/RetinaGuard)

---

## Notebook Overview
This notebook trains and evaluates the **RetinaGuard** architecture:
1. **Multi-Task Objective Formulation**:
   $$\mathcal{L}_{total} = \mathcal{L}_{CE}(y, \hat{y}) + \lambda_1 \mathcal{L}_{lesion}(M, \hat{M}) + \lambda_2 \mathcal{L}_{adv}(d, \hat{d}) + \lambda_3 \mathcal{L}_{cov}(Z_{dis}, Z_{dom})$$
2. **Adversarial GRL Warmup**: Stabilizing feature representation prior to domain confusion.
3. **Model Training Execution**: Multi-center training balancing IDRiD and APTOS cohorts.
4. **Systematic 4-Way Component Ablation Study**: Quantifying individual contributions of lesion attention and causal disentanglement.

### Step 1: Environment & Device Inspection

In [ ]:
import sys
from pathlib import Path
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path("..").resolve() if Path("..").resolve().joinpath("src").exists() else Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.baseline import DRClassifier
from src.models.retinaguard import RetinaGuardModel
from src.utils.reproducibility import get_device, set_seed

set_seed(42)
device = get_device()
print(f"Active PyTorch Execution Device: {device}")

### Step 2: Architecture Parameter Inspection
Inspect model complexity and trainable parameter count across standard ERM baseline and the unified RetinaGuard model.

In [ ]:
base_model = DRClassifier(architecture="resnet18", num_classes=5)
rg_model = RetinaGuardModel(architecture="resnet18", num_classes=5, num_domains=2)

base_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
rg_params = sum(p.numel() for p in rg_model.parameters() if p.requires_grad)

print(f"Baseline ResNet-18 Parameters:  {base_params:,}")
print(f"RetinaGuard Unified Parameters: {rg_params:,} (+{(rg_params - base_params)/base_params*100:.1f}% overhead for lesion + causal branches)")

### Step 3: Run Multi-Center RetinaGuard Training Pipeline
Execute training with class-balanced weighting and dynamic adversarial warmup schedule.

In [ ]:
# Execute RetinaGuard training
# Set epochs as desired (e.g. 6 for fast convergence, 30 for full convergence)
!python -m src.training.train_retinaguard --epochs 6 --batch-size 16 --image-size 256 --warmup-epochs 2

### Step 4: Run Systematic Component Ablation Study
Scientifically test the 4 modular configurations across uniform test cohorts:
- **Config 1**: Baseline ResNet-18 (Standard ERM, $\mathcal{L}_{CE}$)
- **Config 2**: ResNet-18 + Spatial Lesion Attention ($+\mathcal{L}_{lesion}$)
- **Config 3**: ResNet-18 + Causal Disentangler ($+\mathcal{L}_{adv} + \mathcal{L}_{cov}$)
- **Config 4**: Unified RetinaGuard (Full multi-task formulation)

In [ ]:
!python -m src.evaluation.run_ablations --max-samples 30

### Step 5: Visualize Scientific Ablation Metrics
Load and display the quantitative ablation trajectory across In-Domain QWK, Cross-Domain Generalization Gap, and Attribution IoU.

In [ ]:
csv_path = PROJECT_ROOT / "outputs" / "reports" / "ablation_study.csv"
fig_path = PROJECT_ROOT / "outputs" / "figures" / "ablation_metrics_comparison.png"

if csv_path.exists():
    ablation_df = pd.read_csv(csv_path)
    print("=== RetinaGuard Systematic Ablation Results ===")
    display(ablation_df)

if fig_path.exists():
    from PIL import Image
    img = Image.open(fig_path)
    plt.figure(figsize=(14, 6), dpi=150)
    plt.imshow(img)
    plt.axis("off")
    plt.title("Scientific Ablation Trajectory Across Diagnostic & Trustworthiness Metrics", fontsize=13, pad=12)
    plt.show()